In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer


dataset_url = '/content/OFFSIDES.csv'  

df = pd.read_csv(dataset_url, on_bad_lines='skip')  


print("First few rows of the dataset:")
print(df.head())


df['drug_rxnorn_id'] = pd.to_numeric(df['drug_rxnorn_id'], errors='coerce')

df['PRR'] = pd.to_numeric(df['PRR'], errors='coerce') 
df['mean_reporting_frequency'] = pd.to_numeric(df['mean_reporting_frequency'], errors='coerce')
df.dropna(inplace=True)


df.drop_duplicates(inplace=True)

df['drug_concept_name'] = df['drug_concept_name'].str.lower().str.replace(r'[^\w\s]', '', regex=True)
df['condition_concept_name'] = df['condition_concept_name'].str.lower().str.replace(r'[^\w\s]', '', regex=True)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

df['drug_concept_name_tokens'] = df['drug_concept_name'].apply(lambda x: tokenizer.encode(x, truncation=True, padding=True))
df['condition_concept_name_tokens'] = df['condition_concept_name'].apply(lambda x: tokenizer.encode(x, truncation=True, padding=True))

columns_to_drop = ['drug_rxnorn_id', 'condition_meddra_id', 'A', 'B', 'C', 'D', 'PRR_error']
df.drop(columns=columns_to_drop, inplace=True)


print("\nCleaned and Tokenized Data (First few rows):")
print(df[['drug_concept_name', 'drug_concept_name_tokens']].head())

df.to_csv('cleaned_offside_data.csv', index=False)

print("\nCleaned data saved as 'cleaned_offside_data.csv'.")

hf_dataset = Dataset.from_pandas(df)

print("\nConverted Hugging Face Dataset (First row):")
print(hf_dataset[0])


/tmp/ipython-input-1523264349.py:9: DtypeWarning: Columns (0,2,4,5,6,7,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_url, on_bad_lines='skip')  # No compression


First few rows of the dataset:
  drug_rxnorn_id        drug_concept_name condition_meddra_id  \
0           4024  ergoloid mesylates, USP            10002034   
1           4024  ergoloid mesylates, USP            10002965   
2           4024  ergoloid mesylates, USP            10013442   
3           4024  ergoloid mesylates, USP            10023126   
4           4024  ergoloid mesylates, USP            10016288   

                   condition_concept_name  A    B   C     D      PRR  \
0                                 Anaemia  6  126  21  1299  2.85714   
1                   Aplasia pure red cell  1  131   1  1319     10.0   
2  Disseminated intravascular coagulation  1  131   6  1314  1.66667   
3                                Jaundice  2  130   7  1313  2.85714   
4                     Febrile neutropenia  1  131   5  1315      2.0   

  PRR_error mean_reporting_frequency  
0   0.45382                 0.045455  
1   1.41126                 0.007576  
2   1.07626                 